# 19 - Qiskit Interoperability

Convert circuits between QuantSDK and Qiskit.

**Concepts:** Interop, to_qiskit, from_qiskit

In [ ]:
import quantsdk as qs

## QuantSDK -> Qiskit

In [ ]:
# Build a circuit in QuantSDK
circuit = qs.Circuit(2, name="bell")
circuit.h(0).cx(0, 1).measure_all()

print("QuantSDK circuit:")
print(f"  Qubits: {circuit.num_qubits}")
print(f"  Gates: {circuit.gate_count}")
print(f"  Depth: {circuit.depth}")
print(f"  Ops: {circuit.count_ops()}")

In [ ]:
# Convert to Qiskit
try:
    qc = circuit.to_qiskit()
    print("Qiskit circuit:")
    print(f"  Type: {type(qc).__name__}")
    print(f"  Qubits: {qc.num_qubits}")
    print(f"  Depth: {qc.depth()}")
    print(qc.draw())
except ImportError:
    print("Qiskit not installed, showing API usage only")
    print("  circuit.to_qiskit() -> qiskit.QuantumCircuit")

## Qiskit -> QuantSDK

In [ ]:
try:
    from qiskit import QuantumCircuit
    
    # Build in Qiskit
    qc = QuantumCircuit(3)
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.measure_all()
    
    # Convert to QuantSDK
    qs_circuit = qs.Circuit.from_qiskit(qc)
    
    print(f"Converted circuit: {qs_circuit.num_qubits} qubits, {qs_circuit.gate_count} gates")
    
    # Run on QuantSDK
    result = qs.run(qs_circuit, shots=1000, seed=42)
    print(f"Result: {result.counts}")
    print(f"Most likely: {result.most_likely}")
except ImportError:
    print("Qiskit not installed. API:")
    print("  qs.Circuit.from_qiskit(qiskit_circuit) -> qs.Circuit")

## Round-Trip Validation

In [ ]:
# Build complex circuit
original = qs.Circuit(3, name="complex")
original.h(0).cx(0, 1).ry(2, 1.23).cz(1, 2).rx(0, 0.45).measure_all()

# Run original
r1 = qs.run(original, shots=2000, seed=42)

try:
    # Round-trip: QuantSDK -> Qiskit -> QuantSDK
    qc = original.to_qiskit()
    roundtrip = qs.Circuit.from_qiskit(qc)
    
    r2 = qs.run(roundtrip, shots=2000, seed=42)
    
    print("Round-trip fidelity check:")
    print(f"  Original gates: {original.gate_count}")
    print(f"  Roundtrip gates: {roundtrip.gate_count}")
    print(f"  Original top: {r1.most_likely}")
    print(f"  Roundtrip top: {r2.most_likely}")
    print(f"  Match: {r1.most_likely == r2.most_likely}")
except ImportError:
    print(f"Original result: {r1.most_likely}")
    print("Round-trip requires Qiskit installed.")